<a href="https://colab.research.google.com/github/RitalAlmutairi/Rital-Almutairi-Elaan-Aljedaani-ML-Deep-Learning-NLP-Applications-Report/blob/feature%2F31-colab-update/Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Import Libraries**

In [2]:
import kagglehub
import pandas as pd
import csv
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dropout

### **Load And Explore The Data**

In [ ]:
# 1 - Load the dataset

path= kagglehub.dataset_download("shanegerami/ai-vs-human-text")
print(path)
files = os.listdir(path)
csv_file_path = os.path.join(path, files[0])
df = pd.read_csv(csv_file_path)
df=df.sample(n=10000, random_state=42)
# 2 - Print the first 5 rows
display('first 5 rows:',df.head())

# 3 - Check the shape
print('\nCheck the shape:',df.shape)

# 4 - Check for missing values
print('\nCheck for missing values:',df.isnull().sum())

# 5 - Check class balance
print('\nCheck class balance:',df['generated'].value_counts())
# 0.0 Human
# 1.0 Ai

# The dataset is imbalanced because one class contains more samples than the other.
# So, F1-score was used as the primary evaluation metric instead of accuracy.

100%|██████████| 350M/350M [00:08<00:00, 44.5MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/shanegerami/ai-vs-human-text/versions/1


'first 5 rows:'

,text,generated
419952,"Real or Fake Feelings\n\n""Imagine being able t...",0.0
478535,Seeking multiple opinions can help you make be...,0.0
133753,"ADDRESS_NAME\n\nFebruary 9, 2011\n\nDear TEACH...",0.0
68011,"Dear, TEACHER_NAME,\n\nTEACHER_NAME I Believe ...",0.0
195939,Do you believe that there is a computer that c...,0.0


### **Prepare the Data**

In [ ]:
from sklearn.model_selection import train_test_split
# 1 - Identify the axes
X = df['text']
y = df['generated']

# 2 - Dividing the data set into train and test
X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.5, random_state=42, stratify=y
)

In [ ]:
"""
Text datasets only: clean the text
"""
import re, nltk
from nltk.corpus import stopwords
# 1 - Download English stop words
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# 2 - Cleaning function
def clean(text):
 text = text.lower()# Converting text to lowercase letters
 text = re.sub(r'[^a-z\s]', '', text) # Delete any non-English character or distance
 words = [w for w in text.split() if w not in stop_words]
 return ' '.join(words)

#  3 - Applying the cleaning function to the data set
df['clean_text'] = df['text'].apply(clean)

"""
Convert text to numbers with TF-IDF
"""
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000) # 1 - Create a tfidf
X_train = tfidf.fit_transform(X_train) # 2 - Learn vocab on train only and convert training data into numbers
X_test = tfidf.transform(X_test) # 3 - apply to test convert training data into numbers

### **Train and Compare Models**

Logistic Regression Model

In [ ]:
# Train the Logistic Regression Model
model1=LogisticRegression()
model1.fit(X_train,y_train)

# Logistic Regression Model Prediction
logis_pred=model1.predict(X_test)

Random Forest Model

In [ ]:
# Train the Random Forest Model
model2=RandomForestClassifier()
model2.fit(X_train,y_train)

# Random Forest Model Prediction
rndm_pred=model2.predict(X_test)

KNN Model

In [ ]:
# Train the KNN Model
model3=KNeighborsClassifier()
model3.fit(X_train,y_train)

# KNN Model Prediction
knn_pred=model3.predict(X_test)

KeyboardInterrupt: 

Accuracy

In [ ]:
print('Logistic Regression Accuracy:',accuracy_score(y_test,logis_pred))
print('\nRandom Forest Accuracy:',accuracy_score(y_test,rndm_pred))
print('\nKNN Accuracy:',accuracy_score(y_test,knn_pred))

Precision

In [ ]:
print('Logistic Regression Precision:',precision_score(y_test,logis_pred))
print('\nRandom Forest Precision:',precision_score(y_test,rndm_pred))
print('\nKNN Precision:',precision_score(y_test,knn_pred))

Recall

In [ ]:
print('Logistic Regression Recall:',recall_score(y_test,logis_pred))
print('\nRandom Forest Recall:',recall_score(y_test,rndm_pred))
print('\nKNN Recall:',recall_score(y_test,knn_pred))

F1-score

In [ ]:
print('Logistic Regression F1-score:',f1_score(y_test,logis_pred))
print('\nRandom Forest F1-score:',f1_score(y_test,rndm_pred))
print('\nKNN F1-score:',f1_score(y_test,knn_pred))

### **Plot A Confusion Matrix For Each Model**

In [ ]:
cm1=confusion_matrix(y_test,logis_pred)
cm2=confusion_matrix(y_test,rndm_pred)
cm3=confusion_matrix(y_test,knn_pred)

The cmap colors used in this heatmap were inspired by
https://matplotlib.org/stable/users/explain/colors/colormaps.html

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(cm1,annot=True,cmap='RdPu')
plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(cm2,annot=True,cmap='PuRd')
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(cm3,annot=True,cmap='BuPu')
plt.title('KNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show

In [ ]:
results = [
    {
        "Model": "Logistic Regression",
        "Accuracy": 0.9664,
        "Precision":  0.9774353329664281,
        "Recall": 0.9952718676122931,
        "F1-score": 0.9976303317535545
    },
    {
        "Model": "Random Forest",
        "Accuracy": 0.9548,
        "Precision": 0.9912126537785588,
        "Recall": 0.9952718676122931,
        "F1-score": 0.9976303317535545
    },
    {
        "Model": "KNN",
        "Accuracy":  0.9164,
        "Precision": 0.9235596120935539,
        "Recall": 0.9952718676122931,
        "F1-score": 0.9790697674418605
    }
]
results_df = pd.DataFrame(results)
print(f"results_df\n{results_df}")

In [ ]:
The_best_model = results_df.loc[results_df["F1-score"].idxmax()]

print(f"The_Best Model:{The_best_model["Model"]}")

### **Neural Network**

In [ ]:
"""first Neural Network"""
model_1 = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(1, activation="sigmoid"), # One neuron for binary classification
])

#  1 - Assembly of the first model
model_1.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

#  2 - Training the first model using the original training data
history_1 = model_1.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))


In [ ]:
"""second Neural Network"""

# 1 - Convert labels to One-Hot Encoding
# Because neural networks calculate that 1 and 0 have a order, we did One-Hot Encoding
y_train_oh = to_categorical(y_train, num_classes=2)
y_test_oh = to_categorical(y_test, num_classes=2)

model_2 = Sequential([
    Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3), # Arrest30% to reduce overfitting
    Dense(2, activation="softmax"), # two neuron for categorical_crossentropy
])

# 2 - Assembly of the second model
model_2.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

#  3 - Training the first model using the original training data
history_2 = model_2.fit(X_train, y_train_oh, epochs=10, validation_data=(X_test, y_test_oh))

### **Plot the training vs. validation loss and accuracy curves**

The training loss and validation loss both decreased and stayed very close together, which means the model learned well and there are no clear signs of overfitting.

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(history_1.history['loss'])
plt.plot(history_1.history['val_loss'])
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Model 1 Loss')

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(history_2.history['loss'])
plt.plot(history_2.history['val_loss'])
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Model 2 Loss')

The training accuracy and validation accuracy reached very high values and remained close together, indicating that the model performed consistently on both training and validation data.

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(history_1.history['accuracy'])
plt.plot(history_1.history['val_accuracy'])
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Model 1 Accuracy')

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(history_2.history['accuracy'])
plt.plot(history_2.history['val_accuracy'])
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Model 2 Accuracy')

### **Evaluate the neural network using the same metrics as Step 1**

In [ ]:
nn1_pred=model_1.predict(X_test)

In [ ]:
nn2_pred=model_2.predict(X_test)

In [ ]:
nn1_pred_binary=(nn1_pred>0.5).astype(int)
# We used (nn_pred > 0.5).astype(int) to convert the model’s probability outputs (from sigmoid) into final binary classes (0 or 1),
# because the confusion matrix requires discrete predictions instead of probabilities.
print('Neural Network Accuracy:',round(accuracy_score(y_test,nn1_pred_binary),2))
print('\nNeural Network Precision:',round(precision_score(y_test,nn1_pred_binary),2))
print('\nNeural Network Recall:',round(recall_score(y_test,nn1_pred_binary),2))
print('\nNeural Network F1-score:',round(f1_score(y_test,nn1_pred_binary),2))

In [ ]:
nn2_pred_binary=np.argmax(nn2_pred, axis=1)
# We used `np.argmax(nn2_pred, axis=1)` to convert the model’s softmax probability outputs into final class labels
# by selecting the index of the highest probability for each sample, since evaluation metrics require discrete predictions instead of probability distributions.

print('Neural Network Accuracy:',round(accuracy_score(y_test,nn2_pred_binary),2))
print('\nNeural Network Precision:',round(precision_score(y_test,nn2_pred_binary),2))
print('\nNeural Network Recall:',round(recall_score(y_test,nn2_pred_binary),2))
print('\nNeural Network F1-score:',round(f1_score(y_test,nn2_pred_binary),2))

### **Add neural network results to the Step 1 summary table.**

In [ ]:
results.append({
    "Model": "Neural Network 1",
    "Accuracy": 1.0,
    "Precision": 1.0,
    "Recall": 1.0,
    "F1-score": 1.0
})
results.append({
    "Model": "Neural Network 2",
    "Accuracy": 1.0,
    "Precision": 1.0,
    "Recall": 1.0,
    "F1-score": 1.0
})
results

### **Comparison Between Neural Network and Machine Learning Models.**


*   The neural network made fewer classification errors compared to the machine learning models.
*   The neural network model is far superior to all traditional machine learning models, achieving a perfect score of 100% across all metrics.